In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.gold.dim_ciudad")

In [0]:
spark.sql(
    """
    create table if not exists workspace.gold.dim_ciudad(
        id_ciudad bigint,
        pais string,
        departamento string,
        ciudad string,
        lat double,
        lng double,
        poblacion double
    )
   
    """
)

In [0]:
from pyspark.sql import functions as F
df_dim_ciudad=spark.table("workspace.silver.tbl_ciudades")
df_dim_ciudad=(df_dim_ciudad
                      .dropDuplicates(["pais","departamento","ciudad"])
                      .withColumn("id_ciudad",F.monotonically_increasing_id()+1)
                      .select("id_ciudad","pais","departamento","ciudad","lat","lng","poblacion")
                      )


In [0]:
df_dim_ciudad.write.format("delta")\
    .option("mergeShema","true")\
    .mode("overwrite")\
    .saveAsTable("workspace.gold.dim_ciudad")

In [0]:
%sql
select distinct ciudad from workspace.gold.dim_ciudad